# parents-dict-by-argidx — worked example 2: Leading and trailing scalar args use original indices

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `parents-dict-by-argidx`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

When scalars appear at the beginning or end of an argument list, the surviving Tensor indices do not start at 0 or end at `len(args)-1`. The parents dict correctly records the absolute positional index for each Tensor, not a compressed count. For example, in `(5, tensor_a, True, tensor_b, 0.1)`, the parents dict is `{1: tensor_a, 3: tensor_b}` — indices 0, 2, 4 simply do not appear.

## Worked solution

**Step 1 — Enumerate all five args.** Positions 0 through 4 map to `5`, `tensor_a`, `True`, `tensor_b`, `0.1`. The dict comprehension visits all five pairs.

**Step 2 — isinstance filter.** `isinstance(5, MiniTensor)` → False. `isinstance(tensor_a, MiniTensor)` → True. `isinstance(True, MiniTensor)` → False. `isinstance(tensor_b, MiniTensor)` → True. `isinstance(0.1, MiniTensor)` → False.

**Step 3 — Result.** The dict has exactly two entries: `{1: tensor_a, 3: tensor_b}`. The first key is 1, not 0.

**Step 4 — Why it matters.** If we had re-numbered the surviving Tensors as `{0: tensor_a, 1: tensor_b}`, the backward dispatch would look up `(func, 0)` and `(func, 1)` — matching back-functions for args at positions 1 and 3 respectively — which is wrong. The absolute indices must be preserved.

In [ ]:
import torch as t
from dataclasses import dataclass
from typing import Any

@dataclass
class MiniTensor:
    array: Any
    grad: Any = None

def build_parents_preserve_idx(args: tuple) -> dict:
    return {idx: a for idx, a in enumerate(args) if isinstance(a, MiniTensor)}

# Scenario: leading int, trailing float, bool in the middle
t.manual_seed(11)
ta = MiniTensor(t.randn(4))
tb = MiniTensor(t.randn(4))

args_border = (5, ta, True, tb, 0.1)
parents = build_parents_preserve_idx(args_border)

print(f'Keys: {sorted(parents.keys())}')        # [1, 3]
print(f'parents[1] is ta: {parents[1] is ta}')  # True
print(f'parents[3] is tb: {parents[3] is tb}')  # True
print(f'No key 0: {0 not in parents}')           # True
print(f'No key 2: {2 not in parents}')           # True
print(f'No key 4: {4 not in parents}')           # True
print(f'Total entries: {len(parents)}')          # 2